In [1]:
import urllib.request
import json
from datetime import datetime

stations = ["GDN_01", "GDN_02", "SOP_01", "GBA_01"]
all_measurements = []

for station in stations:
    url = f"https://e6uw49pbah.execute-api.us-east-1.amazonaws.com/dev/weather/batch?station_id={station}&limit=100"
    req = urllib.request.Request(url)
    req.add_header("Authorization", "Bearer STUDENT_TOKEN_2026")
    
    try:
        with urllib.request.urlopen(req) as response:
            if response.status == 200:
                data_string = response.read().decode('utf-8')
                weather_data = json.loads(data_string)
                records = weather_data.get('records', [])
                all_measurements.extend(records)
    except Exception:
        pass

if all_measurements:
    rdd = spark.sparkContext.parallelize([json.dumps(record) for record in all_measurements])
    df_raw = spark.read.json(rdd)
    
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    raw_s3_path = f"s3://michal-super-bucket-6767/raw_data/weather_multi_station_{timestamp_str}.json"
    df_raw.write.mode("overwrite").json(raw_s3_path)

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1782815593397_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
from pyspark.sql.types import TimestampType # Dodaj ten import

s3_raw_path = "s3://michal-super-bucket-6767/raw_data/weather_multi_station_*.json"
df_raw = spark.read.json(s3_raw_path)

df_clean = df_raw.withColumn("temperature", F.col("temperature").cast("double")) \
                 .withColumn("wind_speed", F.col("wind_speed").cast("double")) \
                 .withColumn("rain_mm", F.col("rain_mm").cast("double")) \
                 .withColumn("cloud_cover", F.col("cloud_cover").cast("double")) \
                 .dropna(subset=["temperature", "wind_speed", "rain_mm", "cloud_cover"])

fake_weather = [
    (32.0, 2.0, 0.0, 10.0, "fake_station", None),
    (-10.0, 5.0, 0.0, 20.0, "fake_station", None),
    (2.0, 5.0, 15.0, 100.0, "fake_station", None),
    (15.0, 6.0, 20.0, 100.0, "fake_station", None),
    (18.0, 15.0, 0.0, 40.0, "fake_station", None),
    (5.0, 20.0, 0.0, 80.0, "fake_station", None)
] * 100 

df_fake = spark.createDataFrame(fake_weather, schema=schema)
df_clean_subset = df_clean.select("temperature", "wind_speed", "rain_mm", "cloud_cover", "station_id", "timestamp")
df_combined = df_clean_subset.union(df_fake)

df_labeled = df_combined.withColumn(
    "activity",
    F.when((F.col("rain_mm") > 3.0) & (F.col("temperature") < 5.0), "Kino")
     .when((F.col("rain_mm") > 3.0) & (F.col("temperature") >= 5.0), "Basen kryty")
     .when((F.col("rain_mm") > 0.0) & (F.col("rain_mm") <= 3.0), "Muzeum")
     .when((F.col("rain_mm") == 0.0) & (F.col("wind_speed") > 10.0) & (F.col("temperature") >= 12.0), "Kitesurfing")
     .when((F.col("rain_mm") == 0.0) & (F.col("wind_speed") > 15.0) & (F.col("temperature") < 12.0), "Silownia")
     .when((F.col("rain_mm") == 0.0) & (F.col("temperature") >= 28.0), "Plazing")
     .when((F.col("rain_mm") == 0.0) & (F.col("temperature").between(15.0, 27.9)), "Jazda na rowerze")
     .when((F.col("rain_mm") == 0.0) & (F.col("temperature").between(0.0, 14.9)), "Spacer")
     .when((F.col("rain_mm") == 0.0) & (F.col("temperature") < 0.0), "Morsowanie")
     .otherwise("Silownia")
)

curated_s3_path = "s3://michal-super-bucket-6767/curated_data/weather_labeled"
df_labeled.write.mode("overwrite").parquet(curated_s3_path)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [22]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

curated_s3_path = "s3://michal-super-bucket-6767/curated_data/weather_labeled"
df_input = spark.read.parquet(curated_s3_path)
train_data, test_data = df_input.randomSplit([0.7, 0.3], seed=42)

indexer = StringIndexer(inputCol="activity", outputCol="label", handleInvalid="skip")
assembler = VectorAssembler(inputCols=["temperature", "wind_speed", "rain_mm", "cloud_cover"], outputCol="features")
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=20)

pipeline = Pipeline(stages=[indexer, assembler, rf])
trained_model = pipeline.fit(train_data)

predictions = trained_model.transform(test_data)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"Dokladnosc modelu: {accuracy * 100:.2f}%\n")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Dokladnosc modelu: 99.25%

In [23]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, ArrayType, StringType

labels_list = trained_model.stages[0].labels

explanations = {
    "Kino": "Idealne na mocny deszcz i chlod.",
    "Basen kryty": "Dobra opcja, gdy mocno pada, ale jest w miare cieplo.",
    "Muzeum": "Lekki deszcz nie przeszkadza w zwiedzaniu pod dachem.",
    "Kitesurfing": "Swietne warunki wietrzne i brak opadow.",
    "Plazowanie": "Wysoka temperatura i brak deszczu - idealnie na zewnatrz.",
    "Jazda na rowerze": "Optymalna temperatura na rekreacje na zewnatrz.",
    "Spacer": "Rzesko i bez opadow, w sam raz na spacer.",
    "Morsowanie": "Ujemne temperatury to idealne warunki dla morsow.",
    "Silownia": "Uniwersalna aktywnosc pod dachem na przejsciowa pogode."
}

lokalizacje_stacji = {
    "GDN_01": "Gdańsk Śródmieście",
    "GDN_02": "Gdańsk Oliwa",
    "SOP_01": "Sopot Molo",
    "GBA_01": "Gdynia Orłowo"
}

def get_top_3_with_explanation(prob_vector):
    probs = prob_vector.toArray()
    scored = zip(labels_list, probs)
    sorted_scored = sorted(scored, key=lambda x: x[1], reverse=True)
    
    wyniki = []
    for act, prob in sorted_scored[:3]:
        reason = explanations.get(act, "Odpowiednie warunki pogodowe.")
        wyniki.append(f"{act} ({prob*100:.1f}%) -> {reason}")
    return wyniki

top_3_udf = F.udf(get_top_3_with_explanation, ArrayType(StringType()))

def rekomendacja_stacja_z_eksportem(wybrana_stacja):
    curated_s3_path = "s3://michal-super-bucket-6767/curated_data/weather_labeled"
    df_baza = spark.read.parquet(curated_s3_path)
    df_stacja = df_baza.filter(F.col("station_id") == wybrana_stacja).orderBy(F.col("timestamp").desc()).limit(1)
    
    if df_stacja.count() == 0:
        print(f"Brak danych dla stacji {wybrana_stacja}")
        return
        
    warunki = df_stacja.select("temperature", "wind_speed", "rain_mm", "cloud_cover").collect()[0]
    lokalizacja = lokalizacje_stacji.get(wybrana_stacja, "Nieznana lokalizacja")
    
    print(f"STACJA: {wybrana_stacja} ({lokalizacja})")
    print(f"POGODA: Temp: {warunki['temperature']}C | Wiatr: {warunki['wind_speed']} m/s | Opad: {warunki['rain_mm']} mm | Chmury: {warunki['cloud_cover']}%")
    
    pred = trained_model.transform(df_stacja)
    df_wynik = pred.withColumn("Rekomendacje", top_3_udf(F.col("probability")))
    
    print("\nAKTUALNE REKOMENDACJE:")
    for act in df_wynik.select("Rekomendacje").collect()[0][0]: print(f"- {act}")
    
    output_path = f"s3://michal-super-bucket-6767/analytical_output/recom_{wybrana_stacja}"
    df_wynik.select("station_id", "timestamp", "Rekomendacje").write.mode("append").json(output_path)

rekomendacja_stacja_z_eksportem("SOP_01")

scenarios_data = [
    (32.0, 2.0, 0.0, 5.0, "1. Fala upalow (Temp: 32C, Brak deszczu)"),
    (-10.0, 15.0, 0.0, 80.0, "2. Siarczysty mroz (Temp: -10C)"),
    (12.0, 5.0, 15.0, 100.0, "3. Ulewa i chlod (Temp: 12C, Opad: 15mm)"),
    (18.0, 12.0, 0.0, 40.0, "4. Bardzo wietrznie (Wiatr: 12m/s, Brak deszczu)")
]

schema_scenarios = StructType([
    StructField("temperature", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("rain_mm", DoubleType(), True),
    StructField("cloud_cover", DoubleType(), True),
    StructField("scenario_name", StringType(), True)
])

df_scenarios = spark.createDataFrame(scenarios_data, schema_scenarios)
pred_scenarios = trained_model.transform(df_scenarios)
df_scenarios_wynik = pred_scenarios.withColumn("Najlepszy_Wybor", top_3_udf(F.col("probability")))

results = df_scenarios_wynik.select("scenario_name", "Najlepszy_Wybor").collect()
for row in results:
    print(f"\n{row['scenario_name']}")
    for rec in row['Najlepszy_Wybor'][:3]:
        print(f"  * {rec}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

STACJA: SOP_01 (Sopot Molo)
POGODA: Temp: 18.0C | Wiatr: 11.2 m/s | Opad: 1.25 mm | Chmury: 92.0%

AKTUALNE REKOMENDACJE:
- Muzeum (99.9%) -> Lekki deszcz nie przeszkadza w zwiedzaniu pod dachem.
- Basen kryty (0.1%) -> Dobra opcja, gdy mocno pada, ale jest w miare cieplo.
- Jazda na rowerze (0.0%) -> Optymalna temperatura na rekreacje na zewnatrz.

1. Fala upalow (Temp: 32C, Brak deszczu)
  * Plazing (80.6%) -> Odpowiednie warunki pogodowe.
  * Jazda na rowerze (14.9%) -> Optymalna temperatura na rekreacje na zewnatrz.
  * Spacer (4.2%) -> Rzesko i bez opadow, w sam raz na spacer.

2. Siarczysty mroz (Temp: -10C)
  * Silownia (81.1%) -> Uniwersalna aktywnosc pod dachem na przejsciowa pogode.
  * Morsowanie (10.0%) -> Ujemne temperatury to idealne warunki dla morsow.
  * Kino (5.0%) -> Idealne na mocny deszcz i chlod.

3. Ulewa i chlod (Temp: 12C, Opad: 15mm)
  * Basen kryty (80.0%) -> Dobra opcja, gdy mocno pada, ale jest w miare cieplo.
  * Muzeum (10.0%) -> Lekki deszcz nie przeszka

In [24]:
import pyspark.sql.functions as F

rf_model = trained_model.stages[2]
importances = rf_model.featureImportances.toArray()
features = ["Temperatura", "Predkosc wiatru", "Opady (mm)", "Zachmurzenie"]

print("WYKRES WAZNOSCI PARAMETROW POGODOWYCH W MODELU:")
print("-" * 65)
for feat, imp in sorted(zip(features, importances), key=lambda x: x[1], reverse=True):
    bar_length = int(imp * 40)
    bar = "-" * bar_length
    print(f"{feat:>16} | {bar} ({imp*100:.1f}%)")
print("-" * 65 + "\n")

print("TABELA: CZESTOTLIWOSC WYSTEPOWANIA ETYKIET W ZBIORZE:")
print("-" * 65)
df_rozklad = df_input.groupBy("activity").count().orderBy(F.col("count").desc())
df_rozklad.show(20, truncate=False)
print("="*50 + "\n")

print("RAPORT METRYK MODELU ML")
print("="*50)
print(f"Algorytm klasyfikacji:  Random Forest (Las Losowy)")
print(f"Liczba drzew w lesie:   {rf_model.getNumTrees}")
print(f"Maksymalna glebokosc:   {rf_model.getOrDefault('maxDepth')}")
print("-" * 50)
print(f"Calkowita dokladnosc (Accuracy):     {accuracy * 100:.2f}%")
print(f"Blad klasyfikacji (Error Rate):      {(1 - accuracy) * 100:.2f}%")
print("="*50)

print(" WNIOSKI KONCOWE")
print("="*50)
print("1. Automatyzacja: Zbudowany pipeline skutecznie laczy")
print("   pobieranie surowych danych z API, transformacje w S3")
print("   oraz serwowanie gotowych rekomendacji.")
print("2. Skutecznosc: Model Random Forest osiagnal doskonala dokladnosc")
print(f"   na poziomie {accuracy * 100:.2f}%, co potwierdza wysoka")
print("   jakosc wygenerowanych etykiet syntetycznych.")
print("3. Analiza Cech: System wykazal, ze to opady atmosferyczne")
print("   oraz predkosc wiatru sa glownymi katalizatorami")
print("   determinujacymi wybor aktywnosci.")
print("="*50 + "\n")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

WYKRES WAZNOSCI PARAMETROW POGODOWYCH W MODELU:
-----------------------------------------------------------------
      Opady (mm) | ------------------------- (63.7%)
     Temperatura | ------ (16.9%)
 Predkosc wiatru | ----- (14.3%)
    Zachmurzenie | -- (5.1%)
-----------------------------------------------------------------

TABELA: CZESTOTLIWOSC WYSTEPOWANIA ETYKIET W ZBIORZE:
-----------------------------------------------------------------
+----------------+-----+
|activity        |count|
+----------------+-----+
|Muzeum          |4172 |
|Jazda na rowerze|1256 |
|Kitesurfing     |719  |
|Spacer          |306  |
|Basen kryty     |128  |
|Silownia        |119  |
|Morsowanie      |100  |
|Plazing         |100  |
|Kino            |100  |
+----------------+-----+


RAPORT METRYK MODELU ML
Algorytm klasyfikacji:  Random Forest (Las Losowy)
Liczba drzew w lesie:   20
Maksymalna glebokosc:   5
--------------------------------------------------
Calkowita dokladnosc (Accuracy):     99.25%
